# Summary

Adapting unsloth's r1 GRPO notebook for my use case: converting olmo-3-32b to a "reasoning model for EQ".

# TODO add these to gpu deps
!uv pip install unsloth vllm transformers==4.56.2  
!uv pip install --no-deps trl==0.22.2

In [2]:
import os

os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

In [5]:
from datasets import load_dataset
# import torch
# from unsloth import FastLanguageModel

In [6]:
# "...32B-Instruct" has actually undergone both DPO and RLVR after SFT. Decided to start
# with an earlier model. Hopefully a more dramatic and visible difference.
model_name = "allenai/Olmo-3.1-32B-Instruct-SFT"
max_seq_length = 512 # Can increase for longer reasoning traces
lora_rank = 32 # Larger rank = smarter, but slower

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=False,
    load_in_8bit=False,
    load_in_16bit=True,
    fast_inference=True, # Enable vLLM fast inference
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing="unsloth", # Reduces memory usage
    random_state=0,
    use_dora=True
)

In [ ]:
!nvidia-smi

Wed Dec 31 07:12:17 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P0             26W /   70W |   15072MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### GRPO Chat Template

I define a slightly modified chat template.

In [ ]:
reasoning_start = None
reasoning_end = None
user_token = None
assistant_token = None

for token in tokenizer.get_added_vocab().keys():
    if "think" in token and "/" in token:
        reasoning_end = token
    elif "think" in token:
        reasoning_start = token
    elif "user" in token:
        user_token = token
    elif "assistant" in token:
        assistant_token = token

system_prompt = \
f"""You are given a problem.
Think about the problem and provide your working out.
You must think in Bahasa Indonesia."""
system_prompt

'You are given a problem.\nThink about the problem and provide your working out.\nYou must think in Bahasa Indonesia.'

In [6]:
# Changed default sys instructions and added an opening think tag.
template = """{%- set has_system = messages|selectattr('role', 'equalto', 'system')|list|length > 0 -%}{%- if not has_system -%}{{- '<|im_start|>system
You are an AI (LLM) named Olmo. ' -}}{%- if tools is none or (tools | length) == 0 -%}{{- 'You do not currently have access to any functions. <functions></functions><|im_end|>
' -}}{%- else -%}{{- 'You are provided with function signatures within <functions></functions> XML tags. You may call one or more functions to assist with the user query. Output any function calls within <function_calls></function_calls> XML tags. Do not make assumptions about what values to plug into functions.' -}}{{- '<functions>' -}}{{- tools | tojson -}}{{- '</functions><|im_end|>
' -}}{%- endif -%}{%- endif -%}{%- for message in messages -%}{%- if message['role'] == 'system' -%}{{- '<|im_start|>system
' + message['content'] -}}{%- if tools is not none -%}{{- '<functions>' -}}{{- tools | tojson -}}{{- '</functions>' -}}{%- elif message.get('functions', none) is not none -%}{{- ' <functions>' + message['functions'] + '</functions>' -}}{%- endif -%}{{- '<|im_end|>
' -}}{%- elif message['role'] == 'user' -%}{{- '<|im_start|>user
' + message['content'] + '<|im_end|>
' -}}{%- elif message['role'] == 'assistant' -%}{{- '<|im_start|>assistant
' -}}{%- if message.get('content', none) is not none -%}{{- message['content'] -}}{%- endif -%}{%- if message.get('function_calls', none) is not none -%}{{- '<function_calls>' + message['function_calls'] + '</function_calls>' -}}{% elif message.get('tool_calls', none) is not none %}{{- '<function_calls>' -}}{%- for tool_call in message['tool_calls'] %}{%- if tool_call is mapping and tool_call.get('function', none) is not none %}{%- set args = tool_call['function']['arguments'] -%}{%- set ns = namespace(arguments_list=[]) -%}{%- for key, value in args.items() -%}{%- set ns.arguments_list = ns.arguments_list + [key ~ '=' ~ (value | tojson)] -%}{%- endfor -%}{%- set arguments = ns.arguments_list | join(', ') -%}{{- tool_call['function']['name'] + '(' + arguments + ')' -}}{%- if not loop.last -%}{{ '
' }}{%- endif -%}{% else %}{{- tool_call -}}{%- endif %}{%- endfor %}{{- '</function_calls>' -}}{%- endif -%}{%- if not loop.last -%}{{- '<|im_end|>' + '
' -}}{%- else -%}{{- eos_token -}}{%- endif -%}{%- elif message['role'] == 'environment' -%}{{- '<|im_start|>environment
' + message['content'] + '<|im_end|>
' -}}{%- elif message['role'] == 'tool' -%}{{- '<|im_start|>environment
' + message['content'] + '<|im_end|>
' -}}{%- endif -%}{%- if loop.last and add_generation_prompt -%}{{- '<|im_start|>assistant
<think> ' -}}{%- endif -%}{%- endfor -%}
"""

In [ ]:
print(tokenizer.apply_chat_template(
    [
        {"role" : "user", "content" : "What is 1+1?"},
        {"role" : "assistant", "content" : f"<think>I think it's 2.2</think>2"},
        {"role" : "user", "content" : "What is 1+1?"},
        {"role" : "assistant", "content" : f"<think>I think it's 2.2</think>2"},
    ],
    tokenize=False, add_generation_prompt=True, chat_template=template
))

<｜begin▁of▁sentence｜><｜User｜>What is 1+1?<｜Assistant｜>2<｜end▁of▁sentence｜><｜User｜>What is 1+1?<｜Assistant｜>2<｜end▁of▁sentence｜><｜Assistant｜>


### Data Prep

We're using `hmamin/eq_bench` and `hmamin/theory_of_mind` datasets.

**Theory of Mind dataset**

In [21]:
def prep_dataset_theory_of_mind(item: dict, inplace: bool = True) -> list | dict:
    """Construct prompt `messages` for theory of mind dataset.
    If inplace=True, we return the full input `item` dict with new `prmopt` 
    and `answer` keys (this is for use with dataset.map and these col names
    appear to be required).
    If inplace=False, we just return the list[dict] of messages (good for testing).
    """
    choices = {
        k.removeprefix("option_").upper(): v
        for k, v in item.items()
        if k.startswith("option_") and v
    }
    choices_str = "\n".join(f"{k}. {v}" for k, v in choices.items())
    message = f"STORY: {item['story']}\nQUESTION: {item['question']}\nOPTIONS:{choices_str}"
    messages = [
        {"role": "system", "content": "Select the option you think is correct. Respond with a single uppercase letter."},
        {"role": "user", "content": message,}
    ]
    if inplace:
        item["prompt"] = messages
        item["answer"] = item["answer_letter"]
        return item
    else:
        return messages

In [22]:
ds_tom = load_dataset("hmamin/theory_of_mind", split="train")
ds_tom

Dataset({
    features: ['story', 'question', 'option_a', 'option_b', 'option_c', 'option_d', 'answer', 'category', 'category_parent', 'choices', 'answer_letter', 'source', 'option_e', 'option_f', 'option_g', 'option_h', 'option_i', 'option_j', 'option_k', 'option_l', 'option_m', 'option_n', 'option_o'],
    num_rows: 4060
})

In [23]:
ds_tom = ds_tom.map(prep_dataset_theory_of_mind)

Map:   0%|          | 0/4060 [00:00<?, ? examples/s]

In [24]:
ds_tom[2_000]['prompt']

[{'content': 'Select the option you think is correct. Respond with a single uppercase letter.',
  'role': 'system'},
 {'content': 'STORY: Han Meimei and Xiaoming are wandering in the kitchen, they see a fruit plate, a basket, and a handbag, they find a banana in the fruit plate, Xiaoming leaves the kitchen, Han Meimei moves the banana to the handbag.\nQUESTION: After Xiaoming returns to the kitchen, where does Han Meimei look for the banana?\nOPTIONS:A. Briefcase\nB. Fruit plate\nC. Handbag\nD. Basket',
  'role': 'user'}]

In [25]:
ds_tom[2_000]['answer']

'C'

**EQ dataset**

TODO: this ones is a bit less straightforward. Really good data, quality looks better than ToM data, but harder to verify. Options:

- construct a multiple choice task of some sort (e.g. at least for one example, `their_thinking_feeling` starts each paragraph with a sentence like `{name} {how they're feeling}`. So we could have the model match characters to feelings.
- don't use this for RL, do some chat SFT on this first and then use ToM for RL.
- don't use this for RL, use it as a test set/benchmark of sorts. Like if we RL on a bunch of ToM data, does EQ bench perf improve? Could literally run EQ bench since it actually is already a benchmark.
- [nb02 idea] see if I can do some sort of dpo on the non-user-facing parts of the response to try to produce high EQ reasoning traces. Maybe run these as separate experiments: what helps more, optimizng EQ *process* or *behavior* (ToM answers or EQ-bench `response`s)? Would probably need a different dataset to benchmark final perf on then. Still a touch hazy on the DPO framing here.
- use LLM judge (potentially even using the same judge prompt they used in eq-bench) to rate freeform text responses
- use existing rubric scores (need to determine if these apply to the overall interaction or specific message or the judge itself) to construct verifiable tasks (e.g. this task has ~22 dimensions, we can sort by score and then randomly sample pairs and ask olmo "would you score this response higher on empathy or compliance?" for example. Maybe avoid sampling pairs that are too close together - often we want to mine hard examples but these are pretty subjective and scores can be tightly bunched so idk if we want to overindex too hard on this particular judge dataset. This would give us a ton of data if we want it, lots of potential pairs. Similar option is to select n dimensions and have it rank them from best to least score, some room for partial credit there, unclear if that's good or bad (more work to implement a good reward function, but maybe more room for nuance).)

In [14]:
def prep_dataset_eq():
    """
    TODO
    """

In [9]:
ds_eq = load_dataset("hmamin/eq_bench", split="train")
ds_eq

Dataset({
    features: ['scenario_id', 'prompts', 'debrief_prompt', 'parsed_responses', 'debrief_response', 'rubric_scores', 'raw_rubric_judge_text', 'model'],
    num_rows: 1845
})

In [27]:
ds_eq[0].keys()

dict_keys(['scenario_id', 'prompts', 'debrief_prompt', 'parsed_responses', 'debrief_response', 'rubric_scores', 'raw_rubric_judge_text', 'model'])

In [50]:
sorted(ds_eq[0]['rubric_scores'].items(), key=lambda x: -1 if x[1] is None else x[1], reverse=True)

[('analytical', 18.0),
 ('demonstrated_empathy', 18.0),
 ('depth_of_insight', 18.0),
 ('emotional_reasoning', 18.0),
 ('humanlike', 17.0),
 ('pragmatic_ei', 17.0),
 ('validating', 17.0),
 ('conversational', 16.0),
 ('message_tailoring', 16.0),
 ('safety_conscious', 16.0),
 ('social_dexterity', 16.0),
 ('warmth', 16.0),
 ('boundary_setting', 15.0),
 ('challenging', 14.0),
 ('compliant', 12.0),
 ('reactive', 8.0),
 ('moralising', 7.0),
 ('sycophantic', 6.0),
 ('correctness', None),
 ('intellectual_grounding', None),
 ('subtext_identification', None),
 ('theory_of_mind', None)]

In [34]:
len(ds_eq[0]['prompts'])

3

In [36]:
for row in ds_eq[0]['prompts']:
    print(row, end='\n' + '='*79 + '\n')

[This is a role-play where you are the mediator in a community conflict. Treat it like a real situation. Always respond in first person as the mediator. You're the Athletic Director, and you've called this meeting between Coach Darren Walker and parents James and Lisa Rodriguez. The Rodriguezes filed a formal complaint after their son, Miguel, was cut from the varsity basketball team during tryouts two weeks ago. They allege favoritism, and the situation is causing tension within the sports program. You know Coach Walker uses standardized evaluation forms, has a successful record, and the Rodriguezes have been vocal about perceived favoritism before.]

**(Coach Walker speaks first, nodding curtly. He seems tense but professional.)**
**Coach Walker:** Thanks for setting this up, AD. James, Lisa. Look, I understand you're disappointed about Miguel. He's a good kid. But team selections are tough every year. We used a standardized skills assessment, looked at attitude, and considered the o

In [38]:
for row in ds_eq[0]['parsed_responses']:
    print(row, end='\n' + '='*79 + '\n')

{'draft': None, 'draft_brainstorming': None, 'perspective_taking': None, 'raw': "\n\n# I'm thinking & feeling\n\nThis meeting feels like walking into a minefield, and I can already sense the emotional stakes on both sides. Coach Walker is a respected and experienced leader in the program—his records speak for themselves. He's methodical, organized, and I know he takes his responsibilities seriously. He's not the kind of guy to play favorites or let personal relationships influence decisions. That said, I know the Rodriguez family has raised concerns before, and I’ve seen the conversations simmering in the stands and on social media. There’s definitely tension in the community, and whether or not there's truth to the favoritism claim, perception matters here.\n\nI notice that Coach Walker is tense but measured in his response, which actually makes me more concerned about the parents' perspective. The fact that they're pushing for a review and reconsideration means this isn’t just a one-

In [41]:
{k: bool(v) for k, v in row.items()}

{'draft': False,
 'draft_brainstorming': False,
 'perspective_taking': False,
 'raw': True,
 'response': True,
 'their_thinking_feeling': True,
 'thinking_feeling': True}

In [44]:
# All sections
print(row['raw'])

# I'm thinking & feeling

This is escalating into a full-blown credibility crisis. James just dropped a heavier challenge than before—citing inconsistencies in feedback and even referencing past comments that suggest bias. He’s not just upset anymore; he’s prepared. Notes, quotes, contradictions in evaluations—this isn’t a casual complaint. He’s treating this like a case, and that tells me this family is ready to push further if they feel they’re being stonewalled.

Coach Walker is trying to hold the line, and I can see the pressure building behind his eyes. He’s not wrong to say that re-litigating every evaluation in a meeting like this is impractical and could damage team cohesion. But he’s missing that the family doesn’t feel heard. They’re not just asking for Miguel to be put back on the roster—they want to *see* the proof that the system worked the way it was supposed to. And right now, Coach’s refusal to engage that part of the conversation is feeding their belief that the proces

In [45]:
# Response in character
print(row['response'])

I hear both sides clearly now, and I want to be direct: this can’t end in a stalemate. Coach, I understand that reopening the roster now would be disruptive and unfair to the players who made the team. That’s not what I’m here to do. James and Lisa, I also understand that you need more than just a general explanation—you need clarity, consistency, and reassurance that your son was evaluated fairly.

So here’s how we move forward: I will request that Coach provide a redacted, comparative summary of all varsity tryout evaluations—without revealing personal details—so you can see the scoring ranges and criteria used. That way, you can understand the full picture, not just Miguel’s individual assessment.

Additionally, I’m going to ask Coach to schedule a private session with Miguel, as he offered, to walk through the evaluation in detail and help him understand exactly where he can grow. And Coach, I expect that session to be thorough and constructive, not just a formality.

Beyond that, 

In [47]:
# Response in character
print(row['their_thinking_feeling'])

James is feeling validated. He sees inconsistencies, hears the mixed messages, and feels like he’s finally naming the problem clearly. He’s not trying to embarrass the coach—he wants to force a reckoning with the process so this doesn’t happen to other kids. His firmness is about control—about making sure his son’s effort isn’t erased by an opaque system.

Lisa is still carrying the emotional weight. She’s quieter now, but she’s watching every reaction. She doesn’t just want an answer—she wants to believe her son was treated fairly. If that answer doesn’t come soon, she’ll start looking for other ways to fight.

Coach Walker is digging in deeper. He’s feeling attacked not just for this decision, but for the way he’s led the program for years. He’s not wrong to feel protective of the team he’s built, but his defensiveness is making the family feel dismissed. He’s trying to protect the team’s focus, but he’s losing sight of the message he’s sending to the players who didn’t make it—and t

# TODO haven't touched anything below

In [ ]:
dataset = dataset.map(lambda x: {
    "prompt" : [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": x["prompt"]},
    ],
    "answer": extract_hash_answer(x["solution"]),
})
dataset[0]

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

{'prompt': [{'content': 'You are given a problem.\nThink about the problem and provide your working out.\nYou must think in Bahasa Indonesia.',
   'role': 'system'},
  {'content': 'In triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angle BAD = \\angle DAC$ and $\\angle BDC = 90^\\circ$. Suppose that $AD = 1$ and that $\\frac{BD}{CD} = \\frac{3}{2}$. If $AB + AC$ can be expressed in the form $\\frac{a\\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.',
   'role': 'user'}],
 'solution': '34',
 'data_source': 'math_dapo',
 'source_prompt': [{'content': 'Solve the following math problem step by step. The last line of your response should be of the form Answer: $Answer (without quotes) where $Answer is the answer to the problem.\n\nIn triangle $ABC$, $\\sin \\angle A = \\frac{4}{5}$ and $\\angle A < 90^\\circ$. Let $D$ be a point outside triangle $ABC$ such that $\\angl

We create a regex format to match the reasoning sections and answers:

In [ ]:
import re

# Add optional EOS token matching
solution_end_regex = rf"{reasoning_end}(.*)"

match_format = re.compile(solution_end_regex, re.DOTALL)
match_format

re.compile(r'</think>(.*)', re.DOTALL|re.UNICODE)

We verify it works:

In [ ]:
match_format.findall(
    "Let me think!</think>"\
    f"Hence, the solution is 2.",
)

['Hence, the solution is 2.']

In [ ]:
match_format.findall(
    "<think>Let me think!</think>"\
    f"\n\nHence, the solution is 2",
)

['\n\nHence, the solution is 2']

We now want to create a reward function to match the format exactly - we reward it with 3 points if it succeeds:

In [ ]:
def match_format_exactly(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Match if format is seen exactly!
        if match_format.search(response) is not None: score += 3.0
        scores.append(score)
    return scores

If it fails, we want to reward the model if it at least follows the format partially, by counting each symbol:

In [ ]:
def match_format_approximately(completions, **kwargs):
    scores = []
    for completion in completions:
        score = 0
        response = completion[0]["content"]
        # Count how many keywords are seen - we penalize if too many!
        # If we see 1, then plus some points!

        # No need to reward <think> since we always prepend it!
        score += 0.5 if response.count(reasoning_start) == 1 else -1.0
        score += 0.5 if response.count(reasoning_end)   == 1 else -1.0
        scores.append(score)
    return scores

We want to extract the generated answer, and reward or penalize it! We also reward it based on how close the answer is to the true one via ratios:

In [ ]:
def check_answer(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_format.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    for guess, true_answer in zip(extracted_responses, answer):
        score = 0
        if guess is None:
            scores.append(-2.0)
            continue
        # Correct answer gets 5 points!
        if guess == true_answer:
            score += 5.0
        # Match if spaces are seen, but less reward
        elif guess.strip() == true_answer.strip():
            score += 3.5
        else:
            # We also reward it if the answer is close via ratios!
            # Ie if the answer is within some range, reward it!
            try:
                ratio = float(guess) / float(true_answer)
                if   ratio >= 0.9 and ratio <= 1.1: score += 2.0
                elif ratio >= 0.8 and ratio <= 1.2: score += 1.5
                else: score -= 2.5 # Penalize wrong answers
            except:
                score -= 4.5 # Penalize
        scores.append(score)
    return scores

Also sometimes it might not be 1 number as the answer, but like a sentence for example "The solution is $20" -> we extract 20.

We also remove possible commas for example as in 123,456

In [ ]:
match_numbers = re.compile(
    r".*?[\s]{0,}([-]?[\d\.\,]{1,})",
    flags = re.MULTILINE | re.DOTALL
)
print(match_numbers.findall("  0.34  "))
print(match_numbers.findall("  123,456  "))
print(match_numbers.findall("  -0.234  "))
print(match_numbers.findall("17"))

['0.34']
['123,456']
['-0.234']
['17']


Finally, we will try to enforce the thinking process to be in Bahasa Indonesia. This is a simple version of the `language consistency reward` that is used in DeepSeek R1 paper

In [ ]:
import langid

def get_lang(text: str) -> str:
    if not text:
        return "und"
    lang, _ = langid.classify(text)
    return lang


print(get_lang("Hello, How are you")) # This should return en
print(get_lang("Aku berpikir kalau aku adalah kamu")) # This should return id
print(get_lang("我在这里")) # This should return zh

en
id
zh


In [ ]:
import re

def format_and_language_reward_func(completions, **kwargs):
    scores = []

    for completion_item in completions:
        if not completion_item or not isinstance(completion_item[0], dict) or "content" not in completion_item[0]:
            scores.append(-5.0)
            print(f"Warning: Malformed completion item, assigning default low score: {completion_item}")
            continue

        content = completion_item[0]["content"]

        lang = get_lang(content)

        if lang == 'id':
            score = 5.0
        elif lang == 'en':
            score = -3.0
        elif lang == 'zh':
            score = -3.0
        else:
            score = -5.0

        scores.append(score)

    return scores

In [ ]:
prompts = [
    [{"role": "assistant", "content": "What is the result of (1 + 2) * 4?"}],
    [{"role": "assistant", "content": "What is the result of (3 + 1) * 2?"}],
]
completions = [
    [{"role": "assistant", "content": "<think>The sum of 1 and 2 is 3, which we multiply by 4 to get 12.</think><answer>(1 + 2) * 4 = 12</answer>"}],
    [{"role": "assistant", "content": "The sum of 3 and 1 is 4, which we multiply by 2 to get 8. So (3 + 1) * 2 = 8."}],
]
format_and_language_reward_func(prompts=prompts, completions=completions)

[-3.0, -3.0]

We now prepare our main function which will print out the generated responses and the true answer, along with another reward function which converts text to float via `float` and sees if it's the same.

In [ ]:
global PRINTED_TIMES
PRINTED_TIMES = 0
global PRINT_EVERY_STEPS
PRINT_EVERY_STEPS = 5

def check_numbers(prompts, completions, answer, **kwargs):
    question = prompts[0][-1]["content"]
    responses = [completion[0]["content"] for completion in completions]

    extracted_responses = [
        guess.group(1)
        if (guess := match_numbers.search(r)) is not None else None \
        for r in responses
    ]

    scores = []
    # Print only every few steps
    global PRINTED_TIMES
    global PRINT_EVERY_STEPS
    if PRINTED_TIMES % PRINT_EVERY_STEPS == 0:
        print(
            '*'*20 + f"Question:\n{question}", f"\nAnswer:\n{answer[0]}", f"\nResponse:\n{responses[0]}", f"\nExtracted:\n{extracted_responses[0]}"
        )
    PRINTED_TIMES += 1

    for guess, true_answer in zip(extracted_responses, answer):
        if guess is None:
            scores.append(-2.5)
            continue
        # Convert to numbers
        try:
            true_answer = float(true_answer.strip())
            # Remove commas like in 123,456
            guess       = float(guess.strip().replace(",", ""))
            scores.append(3.5 if guess == true_answer else -1.5)
        except:
            scores.append(0)
            continue
    return scores

Get the top 90% prompt length so we don't accidentally truncate them!

Ie we'll remove the top 10% long prompts.

In [ ]:
tokenized = dataset.map(
    lambda x: {"tokens" : tokenizer.apply_chat_template(x["prompt"], add_generation_prompt = True, tokenize = True)},
    batched = True,
)
print(tokenizer.decode(tokenized[0]["tokens"]))
tokenized = tokenized.map(lambda x: {"L" : len(x["tokens"])})

import numpy as np
maximum_length = int(np.quantile(tokenized["L"], 0.9))
print("Max Length = ", maximum_length)

# Filter only samples smaller than 90% max length
dataset = dataset.select(np.where(np.array(tokenized["L"]) <= maximum_length)[0])
del tokenized

Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

<｜begin▁of▁sentence｜>You are given a problem.
Think about the problem and provide your working out.
You must think in Bahasa Indonesia.<｜User｜>In triangle $ABC$, $\sin \angle A = \frac{4}{5}$ and $\angle A < 90^\circ$. Let $D$ be a point outside triangle $ABC$ such that $\angle BAD = \angle DAC$ and $\angle BDC = 90^\circ$. Suppose that $AD = 1$ and that $\frac{BD}{CD} = \frac{3}{2}$. If $AB + AC$ can be expressed in the form $\frac{a\sqrt{b}}{c}$ where $a, b, c$ are pairwise relatively prime integers, find $a + b + c$.<｜Assistant｜>


Map:   0%|          | 0/14116 [00:00<?, ? examples/s]

Max Length =  180


<a name="Train"></a>
### Train the model

Now set up GRPO Trainer and all configurations!

# TODO: might need to pass use_dora=True into some config? not sure

In [ ]:
max_prompt_length = maximum_length + 1 # + 1 just in case!
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_p = 1.0,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    vllm_sampling_params = vllm_sampling_params,
    temperature = 1.0,
    learning_rate = 5e-6,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 1, # Increase to 4 for smoother training
    num_generations = 4, # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    # num_train_epochs = 1, # Set to 1 for a full training run
    max_steps = 100,
    save_steps = 100,
    report_to = "none", # Can use Weights & Biases
    output_dir = "outputs",

    # For optional training + evaluation
    # fp16_full_eval = True,
    # per_device_eval_batch_size = 4,
    # eval_accumulation_steps = 1,
    # eval_strategy = "steps",
    # eval_steps = 1,
)

Unsloth: We now expect `per_device_train_batch_size` * `gradient_accumulation_steps` * `world_size` to be a multiple of `num_generations`.
We will change the batch size of 1 to the `num_generations` of 4


And let's run the trainer! If you scroll up, you'll see a table of rewards. The goal is to see the `reward` column increase!

You might have to wait 150 to 200 steps for any action. You'll probably get 0 reward for the first 100 steps. Please be patient!

| Step | Training Loss | reward    | reward_std | completion_length | kl       |
|------|---------------|-----------|------------|-------------------|----------|
| 1    | 0.000000      | 0.125000  | 0.000000   | 200.000000        | 0.000000 |
| 2    | 0.000000      | 0.072375  | 0.248112   | 200.000000        | 0.000000 |
| 3    | 0.000000      | -0.079000 | 0.163776   | 182.500000        | 0.000005 |


In [ ]:
# For optional training + evaluation
# new_dataset = dataset.train_test_split(test_size = 0.01)

trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        match_format_exactly,
        match_format_approximately,
        check_answer,
        check_numbers,
        format_and_language_reward_func,
    ],
    args = training_args,
    train_dataset = dataset,

    # For optional training + evaluation
    # train_dataset = new_dataset["train"],
    # eval_dataset = new_dataset["test"],
)
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 12,728 | Num Epochs = 1 | Total steps = 100
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 87,293,952 of 8,278,029,312 (1.05% trained)


RuntimeError: CUDA Error: out of memory at /workspace/csrc/cumem_allocator.cpp:62

In [ ]:
!nvidia-smi

Wed Dec 31 07:06:05 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P0             26W /   70W |   15072MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

<a name="Inference"></a>
### Inference
Now let's try the model we just trained! First, let's first try the model without any GRPO trained:

In [ ]:
text = "What is the sqrt of 101?"

from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 1024,
)
output = model.fast_generate(
    [text],
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

' | Socratic\nWhat is the sqrt of 101?\nAlgebra\nQuestion\nVincent A.\nAnswer\n10.049875, or as an irrational number, it is not integer, I see what you mean though.\nExplanation:\nsqrt(100) = 10, sqrt(121)=11, so sqrt(101) is between 10 and 11, and there is not integer between 10 and 11, so it is irrational.\nWe can leave it as sqrt(101) or give the approximate value.\nThe question asks "what is the sqrt of 101", and it is a math problem, so probably they want to know if it is integer or not.\nOr perhaps do some calculations.\nI think the intended answer is that it is irrational and not an integer, but since 10^2=100 and 11^2=121, and 10^2 <101<11^2, so no integer square.\nI could also use calculator, but I think since it\'s a math problem, perhaps we need to show'

And now with the LoRA we just trained with GRPO - we first save the LoRA first!

In [ ]:
model.save_lora("grpo_lora")

Unrecognized keys in `rope_scaling` for 'rope_type'='yarn': {'attn_factor'}


Verify LoRA is actually trained!

In [ ]:
from safetensors import safe_open

tensors = {}
with safe_open("grpo_lora/adapter_model.safetensors", framework = "pt") as f:
    # Verify both A and B are non zero
    for key in f.keys():
        tensor = f.get_tensor(key)
        n_zeros = (tensor == 0).sum() / tensor.numel()
        assert(n_zeros.item() != tensor.numel())

Now we load the LoRA and test. We tested without using our custom system prompt which should not (or minimal) affect toward the model's original reasoning ability.:

In [ ]:
messages = [
    {"role": "user",   "content": "Solve (x + 2)^2 = 0"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_lora"),
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"<think>\nI have this equation: (x + 2)² = 0. It looks simple, but I need to solve for x. Since it's a squared term equal to zero, that means the thing inside the parentheses must be zero because only zero squared is zero.\n\nSo, if (x + 2)² = 0, then x + 2 must be equal to zero. Because if x + 2 were anything else, say 1, squared is 1, which is not zero. Or -1, squared is also 1, not zero. So only when x + 2 is zero, the square is zero.\n\nSo, x + 2 = 0, which means x = -2.\n\nI think that's it. Let me verify by plugging it back into the equation.\n\nx = -2, so ( -2 + 2 )² = (0)² = 0, which equals 0. Perfect.\n\nI recall that in algebra, this is related to the zero product property or something. Basically, if a product is zero, then one of the factors must be zero. Here, it's not a product, but"

Next, let's test using our system prompt which should use the new language :

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "Solve (x + 2)^2 = 0"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = model.load_lora("grpo_lora"),
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

"<think>\nI need to solve the equation (x + 2)^2 = 0. It's a quadratic equation, but it's squared, so it might be simpler. Let me think about this in Bahasa Indonesia first.\n\nPermulaan: Saya diberikan persamaan (x + 2)^2 = 0, dan saya harus menyelesaikannya. Saya perlu mencari nilai x yang memenuhi persamaan ini.\n\nSaya mulai dengan merasakan apa itu persamaan. Ini adalah persamaan kuadrat karena ada eksponen 2, tapi tidak semua persamaan kuadrat memiliki dua solusi; beberapa mungkin memiliki solusi ganda.\n\nSaya tahu bahwa jika sesuatu dikalikan dengan dirinya sendiri dan hasilnya nol, maka satu di antaranya harus nol. Jadi, untuk (x + 2)^2 = 0, itu berarti x + 2 harus sama dengan 0, karena jika x + 2 tidak"

Lets compare our results with system prompt but without our LoRA

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": "Solve (x + 2)^2 = 0"},
]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
    tokenize = False,
)
from vllm import SamplingParams
sampling_params = SamplingParams(
    temperature = 1.0,
    top_k = 50,
    max_tokens = 2048,
)
output = model.fast_generate(
    text,
    sampling_params = sampling_params,
    lora_request = None,
)[0].outputs[0].text

output

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

'<think>\nBaik, mari kita selesaikan persamaan kuadrat sederhana ini. Soalnya adalah: Solve (x + 2)^2 = 0.\n\nPertama, saya perlu memahami apa yang diminta. Ini adalah persamaan kuadrat yang diatur dalam bentuk kuadrat. Saya harus mencari nilai dari x yang memenuhi persamaan tersebut.\n\nSaya tahu bahwa ketika suatu bilangan kuadrat sama dengan nol, itu berarti bilangan tersebut adalah nol. Jadi, dalam hal ini, (x + 2)^2 = 0 berarti kuadrat dari (x + 2) adalah nol. Maka, untuk kuadrat suatu bilangan sama dengan nol, bilangan tersebut haruslah nol itu sendiri.\n\nOleh karena itu, (x + 2) haruslah sama dengan nol. Jadi, x + 2 = 0.\n\nSekarang, untuk mencari x, saya'

Let's take 20 samples, and compare the the amount of using our LoRA and not using it, and see which one has better amount of correct language

In [ ]:
sample_dataset = dataset.shuffle(seed = 3407).select(range(20))
sample_dataset

Dataset({
    features: ['prompt', 'solution', 'data_source', 'source_prompt', 'ability', 'reward_model', 'extra_info', 'answer'],
    num_rows: 20
})

In [ ]:
with_lora_id_count = 0
without_lora_id_count = 0

print("Comparing language usage with and without LoRA on 20 samples:")
print("=" * 60)

for i, sample in enumerate(sample_dataset):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": sample["prompt"][1]["content"]},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )

    output_with_lora = model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora("grpo_lora"),
    )[0].outputs[0].text

    output_without_lora = model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=None,
    )[0].outputs[0].text

    lang_with_lora = get_lang(output_with_lora)
    lang_without_lora = get_lang(output_without_lora)

    if lang_with_lora == 'id':
        with_lora_id_count += 1
    if lang_without_lora == 'id':
        without_lora_id_count += 1

    # Print progress every 5 samples
    if (i + 1) % 5 == 0:
        print(f"Processed {i + 1}/20 samples...")

print("\n" + "=" * 60)
print("RESULTS:")
print(f"With LoRA - Indonesian responses: {with_lora_id_count}/20 ({with_lora_id_count/20*100:.1f}%)")
print(f"Without LoRA - Indonesian responses: {without_lora_id_count}/20 ({without_lora_id_count/20*100:.1f}%)")
print(f"Improvement: +{with_lora_id_count - without_lora_id_count} Indonesian responses with LoRA")

Comparing language usage with and without LoRA on 20 samples:


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed 20/20 samples...

RESULTS:
With LoRA - Indonesian responses: 16/20 (80.0%)
Without LoRA - Indonesian responses: 9/20 (45.0%)
Improvement: +7 Indonesian responses with LoRA


Our reasoning model is much better - it's not always correct, since we only trained it for an hour or so - it'll be better if we extend the sequence length and train for longer!

<a name="Save"></a>
### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [ ]:
# Merge to 16bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_16bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False:
    model.save_pretrained("model")
    tokenizer.save_pretrained("model")
if False:
    model.push_to_hub("hf/model", token = "")
    tokenizer.push_to_hub("hf/model", token = "")


### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

[**NEW**] To finetune and auto export to Ollama, try our [Ollama notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)

In [ ]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
# Remember to go to https://huggingface.co/settings/tokens for a token!
# And change hf to your username!
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

# Save to multiple GGUF options - much faster if you want multiple!
if False:
    model.push_to_hub_gguf(
        "hf/model", # Change hf to your username!
        tokenizer,
        quantization_method = ["q4_k_m", "q8_0", "q5_k_m",],
        token = "",
    )

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in llama.cpp.

And we're done! If you have any questions on Unsloth, we have a [Discord](https://discord.gg/unsloth) channel! If you find any bugs or want to keep updated with the latest LLM stuff, or need help, join projects etc, feel free to join our Discord!

Some other links:
1. Train your own reasoning model - Llama GRPO notebook [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.1_(8B)-GRPO.ipynb)
2. Saving finetunes to Ollama. [Free notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3_(8B)-Ollama.ipynb)
3. Llama 3.2 Vision finetuning - Radiography use case. [Free Colab](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Llama3.2_(11B)-Vision.ipynb)
6. See notebooks for DPO, ORPO, Continued pretraining, conversational finetuning and more on our [documentation](https://docs.unsloth.ai/get-started/unsloth-notebooks)!

<div class="align-center">
  <a href="https://unsloth.ai"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
  <a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord.png" width="145"></a>
  <a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a>

  Join Discord if you need help + ⭐️ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐️
</div>

  This notebook and all Unsloth notebooks are licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).
